# Creutz continuum comparison plot

This notebook reads `continuum_summary__*.json` files produced by `analyze_creutz_continuum.py`, rebuilds the full continuum-fit plot, and also plots the extracted continuum-limit estimates as selectable series. Add one entry per fixed-epsilon trajectory in `ANALYSES`, or set `AUTO_DISCOVER_ANALYSES = True` to load every summary below `RESULT_DIR`.

If two continuum runs use the same `r_hat`, flow list, scale, and fit mode, save them in different output directories or rename/copy one result set before rerunning, otherwise the CLI output names collide.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from finalized_analysis_helpers import _ensure_plotly_browser_path

RESULT_DIR = Path('../data/creutz_continuum_p0')

# Set True to use every continuum_summary__*.json below RESULT_DIR.
AUTO_DISCOVER_ANALYSES = False
r_hat = "1p6"

# Add one entry per continuum analysis / fixed-epsilon trajectory.
ANALYSES = [
    {
        'label': 'ε₁ = 0',
        'summary': RESULT_DIR / 'eps1_0' / f'continuum_summary__rhat_{r_hat}__s_0_2_4_6_8__r0__separate_linear__fit_excludes_beta_2p3.json',
    },
    {
        'label': 'ε₁ ∝ a²',
        'summary': RESULT_DIR / 'p0' / f'continuum_summary__rhat_{r_hat}__s_0_2_4_6_8__r0__separate_linear__fit_excludes_beta_2p3.json',
    },
    {
        'label': 'ε₁ ∝ a³',
        'summary': RESULT_DIR / 'p1' / f'continuum_summary__rhat_{r_hat}__s_0_2_4_6_8__r0__separate_linear__fit_excludes_beta_2p3.json',
    },
]

# None means all selected nonzero t_lat values. Use e.g. [0.25, 0.5, 0.75, 1.0].
SELECTED_T_LAT = None
HIDE_ZERO_FLOW = True

# 't_lat' and 't_over_a2' are the same lattice flow coordinate; 'eight_t_over_a2' is kept for old summaries.
X_AXIS = 't_lat'

# Set to tuples like (xmin, xmax) / (ymin, ymax), or leave as None for automatic limits.
XLIM = None
YLIM = None

# Full continuum-fit plot: finite-a points, fit curves, bands, and a=0 markers.
FULL_PLOT_ANALYSIS_INDEX = 0
FULL_XLIM = None
FULL_YLIM = (1.15, 1.55)
SHOW_BETA_LABELS = True
PRINT_FULL_CONTINUUM_VALUES = True
FULL_SAVE_PDF = True
FULL_PDF_PATH = RESULT_DIR / 'creutz_continuum_full_plot.pdf'

# Single-flow comparison plot: all selected analyses at one t_lat value.
SINGLE_FLOW_T_LAT = 0.5
# Set to tuples like (0.0, 0.23) / (1.25, 1.7), or leave as None for automatic limits.
SINGLE_FLOW_XLIM = None
SINGLE_FLOW_YLIM = (1.25, 1.5)
SINGLE_FLOW_SHOW_BETA_LABELS = True
SINGLE_FLOW_SHOW_BANDS = True
PRINT_SINGLE_FLOW_CONTINUUM_VALUES = True
SINGLE_FLOW_SAVE_PDF = True
SINGLE_FLOW_PDF_PATH = RESULT_DIR / 'creutz_continuum_single_flow_comparison.pdf'

# Continuum-limit comparison plot: a=0 values vs t_lat.
SAVE_PDF = True
PDF_PATH = RESULT_DIR / f'creutz_continuum_limits_comparison__rhat_{r_hat}.pdf'

# Plot styling, matching the other Plotly PDF export notebooks.
FIGURE_WIDTH = 950
FIGURE_HEIGHT = 600
COMPARISON_FIGURE_HEIGHT = 720
CAPSIZE = 3
MARKER_SIZE = 7
LINE_WIDTH = 1.7
PDF_MARGIN = {'l': 58, 'r': 78, 't': 12, 'b': 82}
COLOR_SEQUENCE = pio.templates['plotly'].layout.colorway
CONTINUUM_X_AXIS_TITLE = '(a/r₀)²'
CHI_AXIS_TITLE = 'χ̂(r̂)'
CONTINUUM_CHI_AXIS_TITLE = 'χ̂<sub>cont</sub>(r̂)'
CHI2_DOF_AXIS_TITLE = 'χ²/dof'
CONTINUUM_X_HOVER_LABEL = '(a/r<sub>0</sub>)<sup>2</sup>'
CHI_HOVER_LABEL = '&chi;&#770;'

_ensure_plotly_browser_path()
pio.templates.default = 'plotly_white'

In [ ]:
def _finite_float(value):
    if value is None:
        return None
    try:
        out = float(value)
    except (TypeError, ValueError):
        return None
    return out if math.isfinite(out) else None


def _same_float(left, right, *, atol=1e-10):
    return abs(float(left) - float(right)) <= atol


def _row_t_lat(row):
    value = _finite_float(row.get('t_lat'))
    if value is not None:
        return value
    value = _finite_float(row.get('t_over_a2'))
    if value is not None:
        return value
    eight_t = _finite_float(row.get('eight_t_over_a2'))
    return None if eight_t is None else eight_t / 8.0


def _row_eight_t(row):
    value = _finite_float(row.get('eight_t_over_a2'))
    if value is not None:
        return value
    t_lat = _row_t_lat(row)
    return None if t_lat is None else 8.0 * t_lat


def _t_lat_label(t_lat):
    return f't_lat={float(t_lat):g}'


def _t_lat_legend_label(t_lat, suffix=None):
    label = f't<sub>lat</sub>={float(t_lat):g}'
    if suffix:
        label += f' {suffix}'
    return label


def _continuum_t_lat_legend_label(t_lat):
    return f'a=0, t<sub>lat</sub>={float(t_lat):g}'


def _legend_label_with_suffix(label, suffix=None):
    if not suffix:
        return label
    return f'{label} {suffix}'


def _t_lat_is_selected(t_lat):
    if t_lat is None:
        return False
    if HIDE_ZERO_FLOW and _same_float(t_lat, 0.0):
        return False
    if SELECTED_T_LAT is None:
        return True
    return any(_same_float(t_lat, value) for value in SELECTED_T_LAT)


def _selected_specs():
    if AUTO_DISCOVER_ANALYSES:
        return [{'summary': path} for path in sorted(RESULT_DIR.rglob('continuum_summary__*.json'))]
    return list(ANALYSES)


def _default_label(summary, path):
    label = summary.get('eps_label') or path.stem.replace('continuum_summary__', '')
    excluded_betas = summary.get('fit_excluded_betas') or []
    if excluded_betas:
        excluded = ','.join(f'{float(beta):g}' for beta in excluded_betas)
        label = f'{label}; excluded beta={excluded}'
    return label


def _summary_path_from_spec(spec):
    if isinstance(spec, (str, Path)):
        return Path(spec).expanduser(), None
    return Path(spec['summary']).expanduser(), spec.get('label')


def _continuum_rows(summary):
    rows = summary.get('continuum_estimates')
    if rows is None:
        rows = (summary.get('fit') or {}).get('continuum_estimates') or []

    selected = []
    for row in rows:
        t_lat = _row_t_lat(row)
        if not _t_lat_is_selected(t_lat):
            continue
        value = _finite_float(row.get('continuum_value'))
        if value is None:
            continue
        err = _finite_float(row.get('continuum_err'))
        selected.append(
            {
                't_lat': t_lat,
                't_over_a2': t_lat,
                'eight_t_over_a2': _row_eight_t(row),
                'continuum_value': value,
                'continuum_err': np.nan if err is None else err,
                'chi2_dof': _finite_float(row.get('chi2_dof')),
                'n_points': row.get('n_points'),
                'error_source': row.get('continuum_error_source'),
                'fit_mode': row.get('fit_mode'),
            }
        )
    return sorted(selected, key=lambda row: row[X_AXIS])


def load_analyses():
    specs = _selected_specs()
    if not specs:
        raise ValueError('No analyses selected. Add entries to ANALYSES or enable AUTO_DISCOVER_ANALYSES.')

    available = sorted(RESULT_DIR.rglob('continuum_summary__*.json'))
    loaded = []
    seen_labels = {}
    for spec in specs:
        summary_path, explicit_label = _summary_path_from_spec(spec)
        if not summary_path.exists():
            available_text = '\n'.join(f'  - {path}' for path in available) or '  none found'
            raise FileNotFoundError(f'Missing summary file: {summary_path}\nAvailable summaries:\n{available_text}')
        with summary_path.open('r', encoding='utf-8') as handle:
            summary = json.load(handle)
        rows = _continuum_rows(summary)
        if not rows:
            raise ValueError(f'No selected continuum rows found in {summary_path}')
        label = explicit_label or _default_label(summary, summary_path)
        if label in seen_labels:
            seen_labels[label] += 1
            label = f'{label} ({seen_labels[label]})'
        else:
            seen_labels[label] = 1
        loaded.append({'label': label, 'path': summary_path, 'summary': summary, 'rows': rows})
    return loaded


analyses = load_analyses()

print(f'Loaded {len(analyses)} continuum analysis/analyses.')
for index, analysis in enumerate(analyses):
    summary = analysis['summary']
    flows = [row['t_lat'] for row in analysis['rows']]
    print(f'[{index}] {analysis["label"]}')
    print(f'    {analysis["path"]}')
    print(f'    r_hat={summary.get("r_hat")}, scale={summary.get("scale")}, fit_mode={summary.get("fit_mode")}, t_lat={flows}')

## Full continuum-fit plot

Set `FULL_PLOT_ANALYSIS_INDEX` in the first cell to choose which selected analysis is displayed here.

In [ ]:
from analyze_creutz_continuum import (
    design_grid_for_t_lat,
    fit_for_t_lat,
    hex_to_rgba,
    physical_tau_design_grid,
    prediction_band_for_fit,
    sigma_from_row,
    sorted_unique,
)


def _beta_sort_key(value):
    if value is None:
        return math.inf, 'unknown'
    return float(value), f'{float(value):g}'


def _beta_symbol_map(rows):
    symbols = ['circle', 'square', 'diamond', 'triangle-up', 'triangle-down', 'cross', 'x']
    betas = sorted({row.get('beta') for row in rows}, key=_beta_sort_key)
    return {beta: symbols[index % len(symbols)] for index, beta in enumerate(betas)}


def _fmt_value(value, *, precision=8):
    finite = _finite_float(value)
    return 'n/a' if finite is None else f'{finite:.{precision}g}'


def print_full_continuum_values(analysis):
    summary = analysis['summary']
    fit = summary.get('fit') or {}
    rows = _full_plot_rows(summary)
    t_lat_values = sorted_unique([float(_row_t_lat(row)) for row in rows])

    print(f'Continuum results in full plot for {analysis["label"]}:')
    if fit.get('mode') == 'combined-risch':
        value = _fmt_value(fit.get('continuum_value'), precision=10)
        err = _fmt_value(fit.get('continuum_err'), precision=4)
        chi2_dof = _fmt_value(fit.get('chi2_dof'), precision=5)
        n_points = fit.get('n_points', 'n/a')
        error_source = fit.get('continuum_error_source', 'n/a')
        print(f'  shared continuum: chi_hat(a=0) = {value} +/- {err}; chi^2/dof = {chi2_dof}; n = {n_points}; {error_source}')
        return

    for t_lat in t_lat_values:
        subfit = fit_for_t_lat(fit, t_lat)
        value = _fmt_value(subfit.get('continuum_value'), precision=10)
        err = _fmt_value(subfit.get('continuum_err'), precision=4)
        chi2_dof = _fmt_value(subfit.get('chi2_dof'), precision=5)
        n_points = subfit.get('n_points', 'n/a')
        error_source = subfit.get('continuum_error_source', 'n/a')
        print(
            f'  {_t_lat_label(t_lat)}: '
            f'chi_hat(a=0) = {value} +/- {err}; '
            f'chi^2/dof = {chi2_dof}; n = {n_points}; {error_source}'
        )


def _full_plot_rows(summary):
    rows = []
    for row in summary.get('points', []):
        t_lat = _row_t_lat(row)
        if not _t_lat_is_selected(t_lat):
            continue
        if _finite_float(row.get('ahat2')) is None or _finite_float(row.get('chi_hat')) is None:
            continue
        rows.append(row)
    if not rows:
        raise ValueError('No selected finite-a continuum points found. Check HIDE_ZERO_FLOW and SELECTED_T_LAT.')
    return rows


def _x_grid_for_rows(rows, xlim):
    if xlim is not None:
        lo, hi = [float(value) for value in xlim]
        return np.linspace(lo, hi, 240)
    x_max = max(float(row['ahat2']) for row in rows)
    return np.linspace(0.0, max(x_max * 1.05, 1e-12), 240)


def _add_fit_band(fig, x_grid, band, *, color, name, dash=None):
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([x_grid, x_grid[::-1]]),
            y=np.concatenate([band.y_high, band.y_low[::-1]]),
            fill='toself',
            fillcolor=hex_to_rgba(color, 0.16),
            line={'color': 'rgba(0,0,0,0)'},
            hoverinfo='skip',
            showlegend=False,
            name=f'{name} band',
        )
    )
    line = {'color': color, 'width': LINE_WIDTH}
    if dash is not None:
        line['dash'] = dash
    fig.add_trace(
        go.Scatter(
            x=x_grid,
            y=band.y,
            mode='lines',
            line=line,
            name=name,
            hovertemplate=f'{CONTINUUM_X_HOVER_LABEL}=%{{x:.5g}}<br>fit=%{{y:.8g}}<extra>' + name + '</extra>',
        )
    )


def _add_continuum_marker(fig, subfit, *, color, name):
    value = _finite_float(subfit.get('continuum_value'))
    if value is None:
        return
    err = _finite_float(subfit.get('continuum_err'))
    fig.add_trace(
        go.Scatter(
            x=[0.0],
            y=[value],
            error_y={'type': 'data', 'array': [0.0 if err is None else err], 'visible': err is not None, 'thickness': 1, 'width': CAPSIZE},
            mode='markers',
            marker={'symbol': 'square', 'color': color, 'size': MARKER_SIZE + 2},
            name=name,
            customdata=[[np.nan if err is None else err, subfit.get('chi2_dof'), subfit.get('n_points'), subfit.get('continuum_error_source')]],
            hovertemplate=(
                f'{CONTINUUM_X_HOVER_LABEL}=0<br>'
                f'{CHI_HOVER_LABEL}=%{{y:.8g}}<br>'
                'error=%{customdata[0]:.3g}<br>'
                'chi^2/dof=%{customdata[1]:.5g}<br>'
                'fit points=%{customdata[2]}<br>'
                'error source=%{customdata[3]}'
                '<extra>' + name + '</extra>'
            ),
        )
    )


def make_full_continuum_figure(analysis, *, xlim=None, ylim=None):
    summary = analysis['summary']
    fit = summary.get('fit') or {}
    rows = _full_plot_rows(summary)
    x_grid = _x_grid_for_rows(rows, xlim)
    t_lat_values = sorted_unique([float(_row_t_lat(row)) for row in rows])
    beta_symbols = _beta_symbol_map(rows)

    fig = go.Figure()
    for index, t_lat in enumerate(t_lat_values):
        color = COLOR_SEQUENCE[index % len(COLOR_SEQUENCE)]
        subset = [row for row in rows if _same_float(_row_t_lat(row), t_lat)]

        for used_in_fit, point_rows in (
            (True, [row for row in subset if bool(row.get('used_in_fit', True))]),
            (False, [row for row in subset if not bool(row.get('used_in_fit', True))]),
        ):
            if not point_rows:
                continue
            marker_color = color if used_in_fit else '#7f7f7f'
            symbol = [beta_symbols.get(row.get('beta'), 'circle') for row in point_rows] if used_in_fit else 'circle-open'
            name = _t_lat_legend_label(t_lat, 'data') if used_in_fit else _t_lat_legend_label(t_lat, 'excluded')
            fig.add_trace(
                go.Scatter(
                    x=[float(row['ahat2']) for row in point_rows],
                    y=[float(row['chi_hat']) for row in point_rows],
                    error_y={'type': 'data', 'array': [sigma_from_row(row) for row in point_rows], 'visible': True, 'thickness': 1, 'width': CAPSIZE},
                    mode='markers+text' if SHOW_BETA_LABELS else 'markers',
                    text=[f'{float(row["beta"]):g}' if row.get('beta') is not None else '' for row in point_rows],
                    textposition='top center',
                    marker={'color': marker_color, 'size': MARKER_SIZE, 'symbol': symbol},
                    name=name,
                    customdata=[
                        [
                            row.get('beta'),
                            row.get('epsilon_fixed'),
                            row.get('epsilon_inferred'),
                            row.get('t0_over_a2'),
                            _row_t_lat(row),
                            row.get('tau'),
                            row.get('stat_err'),
                            row.get('sys_err'),
                            row.get('total_err'),
                            row.get('used_in_fit', True),
                            row.get('fit_exclusion_reason'),
                        ]
                        for row in point_rows
                    ],
                    hovertemplate=(
                        'beta=%{customdata[0]}<br>'
                        'fixed &epsilon;<sub>1</sub>=%{customdata[1]}<br>'
                        'inferred &epsilon;<sub>1</sub>=%{customdata[2]}<br>'
                        't0/a^2=%{customdata[3]:g}<br>'
                        't_lat=%{customdata[4]:g}<br>'
                        'tau=%{customdata[5]:g}<br>'
                        f'{CONTINUUM_X_HOVER_LABEL}=%{{x:.5g}}<br>'
                        f'{CHI_HOVER_LABEL}=%{{y:.8g}}<br>'
                        'stat=%{customdata[6]:.3g}<br>'
                        'sys=%{customdata[7]:.3g}<br>'
                        'total=%{customdata[8]:.3g}<br>'
                        'used in fit=%{customdata[9]}<br>'
                        'fit exclusion=%{customdata[10]}'
                        '<extra>' + name + '</extra>'
                    ),
                )
            )

        subfit = fit_for_t_lat(fit, t_lat)
        design = design_grid_for_t_lat(fit, x_grid, t_lat)
        band = prediction_band_for_fit(subfit, design)
        _add_fit_band(fig, x_grid, band, color=color, name=_t_lat_legend_label(t_lat, 'fit'))

        if fit.get('mode') != 'combined-risch':
            _add_continuum_marker(fig, subfit, color=color, name=_continuum_t_lat_legend_label(t_lat))

    if fit.get('mode') == 'combined-risch':
        for tau in summary.get('physical_tau_values') or []:
            design = physical_tau_design_grid(x_grid, float(tau))
            band = prediction_band_for_fit(fit, design)
            _add_fit_band(fig, x_grid, band, color='#444444', name=f'physical tau={float(tau):g}', dash='dash')
        _add_continuum_marker(fig, fit, color='#d62728', name='a=0 shared continuum')

    fig.update_layout(
        title=None,
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        xaxis_title=CONTINUUM_X_AXIS_TITLE,
        yaxis_title=CHI_AXIS_TITLE,
        margin=PDF_MARGIN,
        showlegend=True,
        legend={'x': 0.98, 'xanchor': 'right', 'y': 0.6, 'yanchor': 'top', 'font': {'size': 10}},
    )
    fig.update_xaxes(rangemode='tozero', automargin=True, title_standoff=16)
    if xlim is not None:
        fig.update_xaxes(range=list(xlim))
    if ylim is not None:
        fig.update_yaxes(range=list(ylim))
    return fig


full_analysis = analyses[int(FULL_PLOT_ANALYSIS_INDEX)]
print(f'Displaying full continuum plot for analysis {int(FULL_PLOT_ANALYSIS_INDEX)}: {full_analysis["label"]}')
if PRINT_FULL_CONTINUUM_VALUES:
    print_full_continuum_values(full_analysis)
full_fig = make_full_continuum_figure(full_analysis, xlim=FULL_XLIM, ylim=FULL_YLIM)
if FULL_SAVE_PDF:
    FULL_PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    full_fig.write_image(str(FULL_PDF_PATH))
    print(f'Saved PDF: {FULL_PDF_PATH}')
full_fig

## Single-flow comparison plot

This fixes one flow strength and overlays every selected analysis in `ANALYSES` on the same $\hat{\chi}$ vs $(a/r_0)^2$ continuum-extrapolation plot.

In [ ]:
def _open_symbol(symbol):
    return symbol if str(symbol).endswith('-open') else f'{symbol}-open'


def _fmt_value(value, *, precision=8):
    finite = _finite_float(value)
    return 'n/a' if finite is None else f'{finite:.{precision}g}'


def print_single_flow_continuum_values(entries, t_lat):
    print(f'Continuum results for {_t_lat_label(t_lat)}:')
    for entry in entries:
        label = entry['analysis']['label']
        subfit = entry['subfit']
        value = _fmt_value(subfit.get('continuum_value'), precision=10)
        err = _fmt_value(subfit.get('continuum_err'), precision=4)
        chi2_dof = _fmt_value(subfit.get('chi2_dof'), precision=5)
        n_points = subfit.get('n_points', 'n/a')
        error_source = subfit.get('continuum_error_source', 'n/a')
        print(f'  {label}: chi_hat(a=0) = {value} +/- {err}; chi^2/dof = {chi2_dof}; n = {n_points}; {error_source}')


def _rows_for_single_flow(summary, t_lat):
    rows = []
    for row in summary.get('points', []):
        row_t = _row_t_lat(row)
        if row_t is None or not _same_float(row_t, t_lat):
            continue
        if _finite_float(row.get('ahat2')) is None or _finite_float(row.get('chi_hat')) is None:
            continue
        rows.append(row)
    return sorted(rows, key=lambda row: float(row['ahat2']))


def _single_flow_entries(analyses, t_lat):
    entries = []
    skipped = []
    for analysis in analyses:
        summary = analysis['summary']
        fit = summary.get('fit') or {}
        rows = _rows_for_single_flow(summary, t_lat)
        if not rows:
            skipped.append(f'{analysis["label"]}: no finite-a points at {_t_lat_label(t_lat)}')
            continue
        try:
            subfit = fit_for_t_lat(fit, t_lat)
        except KeyError as exc:
            skipped.append(f'{analysis["label"]}: {exc}')
            continue
        entries.append({'analysis': analysis, 'fit': fit, 'subfit': subfit, 'rows': rows})
    for message in skipped:
        print('Skipped:', message)
    if not entries:
        raise ValueError(f'No selected analyses contain {_t_lat_label(t_lat)}.')
    return entries


def _single_flow_x_grid(entries, xlim):
    if xlim is not None:
        lo, hi = [float(value) for value in xlim]
        return np.linspace(lo, hi, 240)
    x_max = max(float(row['ahat2']) for entry in entries for row in entry['rows'])
    return np.linspace(0.0, max(x_max * 1.05, 1e-12), 240)


def make_single_flow_comparison_figure(entries, t_lat, *, xlim=None, ylim=None):
    x_grid = _single_flow_x_grid(entries, xlim)
    beta_symbols = _beta_symbol_map([row for entry in entries for row in entry['rows']])

    fig = go.Figure()
    for index, entry in enumerate(entries):
        analysis = entry['analysis']
        label = analysis['label']
        color = COLOR_SEQUENCE[index % len(COLOR_SEQUENCE)]
        rows = entry['rows']

        for used_in_fit, point_rows in (
            (True, [row for row in rows if bool(row.get('used_in_fit', True))]),
            (False, [row for row in rows if not bool(row.get('used_in_fit', True))]),
        ):
            if not point_rows:
                continue
            base_symbols = [beta_symbols.get(row.get('beta'), 'circle') for row in point_rows]
            symbols = base_symbols if used_in_fit else [_open_symbol(symbol) for symbol in base_symbols]
            trace_name = _legend_label_with_suffix(label) if used_in_fit else _legend_label_with_suffix(label, 'excluded')
            fig.add_trace(
                go.Scatter(
                    x=[float(row['ahat2']) for row in point_rows],
                    y=[float(row['chi_hat']) for row in point_rows],
                    error_y={'type': 'data', 'array': [sigma_from_row(row) for row in point_rows], 'visible': True, 'thickness': 1, 'width': CAPSIZE},
                    mode='markers+text' if SINGLE_FLOW_SHOW_BETA_LABELS else 'markers',
                    text=[f'{float(row["beta"]):g}' if row.get('beta') is not None else '' for row in point_rows],
                    textposition='top center',
                    marker={'color': color, 'size': MARKER_SIZE, 'symbol': symbols},
                    name=trace_name,
                    customdata=[
                        [
                            row.get('beta'),
                            row.get('epsilon_fixed'),
                            row.get('epsilon_inferred'),
                            row.get('t0_over_a2'),
                            _row_t_lat(row),
                            row.get('tau'),
                            row.get('stat_err'),
                            row.get('sys_err'),
                            row.get('total_err'),
                            row.get('used_in_fit', True),
                            row.get('fit_exclusion_reason'),
                        ]
                        for row in point_rows
                    ],
                    hovertemplate=(
                        'beta=%{customdata[0]}<br>'
                        'fixed &epsilon;<sub>1</sub>=%{customdata[1]}<br>'
                        'inferred &epsilon;<sub>1</sub>=%{customdata[2]}<br>'
                        't0/a^2=%{customdata[3]:g}<br>'
                        't_lat=%{customdata[4]:g}<br>'
                        'tau=%{customdata[5]:g}<br>'
                        f'{CONTINUUM_X_HOVER_LABEL}=%{{x:.5g}}<br>'
                        f'{CHI_HOVER_LABEL}=%{{y:.8g}}<br>'
                        'stat=%{customdata[6]:.3g}<br>'
                        'sys=%{customdata[7]:.3g}<br>'
                        'total=%{customdata[8]:.3g}<br>'
                        'used in fit=%{customdata[9]}<br>'
                        'fit exclusion=%{customdata[10]}'
                        '<extra>' + trace_name + '</extra>'
                    ),
                )
            )

        design = design_grid_for_t_lat(entry['fit'], x_grid, float(t_lat))
        band = prediction_band_for_fit(entry['subfit'], design)
        fit_name = _legend_label_with_suffix(label, 'fit')
        if SINGLE_FLOW_SHOW_BANDS:
            _add_fit_band(fig, x_grid, band, color=color, name=fit_name)
        else:
            fig.add_trace(
                go.Scatter(
                    x=x_grid,
                    y=band.y,
                    mode='lines',
                    line={'color': color, 'width': LINE_WIDTH},
                    name=fit_name,
                    hovertemplate=f'{CONTINUUM_X_HOVER_LABEL}=%{{x:.5g}}<br>fit=%{{y:.8g}}<extra>' + fit_name + '</extra>',
                )
            )
        _add_continuum_marker(fig, entry['subfit'], color=color, name=_legend_label_with_suffix(label, 'a=0'))

    fig.update_layout(
        title=None,
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        xaxis_title=CONTINUUM_X_AXIS_TITLE,
        yaxis_title=CHI_AXIS_TITLE,
        margin=PDF_MARGIN,
        showlegend=True,
        legend={'x': 0.98, 'xanchor': 'right', 'y': 0.52, 'yanchor': 'top', 'font': {'size': 10}},
    )
    fig.update_xaxes(rangemode='tozero', automargin=True, title_standoff=16)
    if xlim is not None:
        fig.update_xaxes(range=list(xlim))
    if ylim is not None:
        fig.update_yaxes(range=list(ylim))
    return fig


single_flow_entries = _single_flow_entries(analyses, float(SINGLE_FLOW_T_LAT))
if PRINT_SINGLE_FLOW_CONTINUUM_VALUES:
    print_single_flow_continuum_values(single_flow_entries, SINGLE_FLOW_T_LAT)


single_flow_fig = make_single_flow_comparison_figure(
    single_flow_entries,
    SINGLE_FLOW_T_LAT,
    xlim=SINGLE_FLOW_XLIM,
    ylim=SINGLE_FLOW_YLIM,
)
print(f'Displaying single-flow comparison for {_t_lat_label(float(SINGLE_FLOW_T_LAT))}')
if SINGLE_FLOW_SAVE_PDF:
    SINGLE_FLOW_PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    single_flow_fig.write_image(str(SINGLE_FLOW_PDF_PATH))
    print(f'Saved PDF: {SINGLE_FLOW_PDF_PATH}')
single_flow_fig

## Continuum-limit comparison plot

In [ ]:
def _axis_title():
    if X_AXIS in {'t_lat', 't_over_a2'}:
        return 't<sub>lat</sub>'
    raise ValueError("X_AXIS must be 't_lat' or 't_over_a2'")


def make_creutz_continuum_comparison_figure(analyses, *, xlim=None, ylim=None):
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        row_heights=[0.68, 0.32],
    )
    for index, analysis in enumerate(analyses):
        rows = sorted(analysis['rows'], key=lambda row: row[X_AXIS])
        color = COLOR_SEQUENCE[index % len(COLOR_SEQUENCE)]
        x = np.asarray([row[X_AXIS] for row in rows], dtype=float)
        y = np.asarray([row['continuum_value'] for row in rows], dtype=float)
        yerr = np.asarray([row['continuum_err'] for row in rows], dtype=float)
        visible_yerr = np.isfinite(yerr) & (yerr >= 0)
        chi2_dof = np.asarray(
            [np.nan if row['chi2_dof'] is None else row['chi2_dof'] for row in rows],
            dtype=float,
        )
        visible_chi2 = np.isfinite(chi2_dof)
        customdata = [
            [
                row['t_lat'],
                row['continuum_err'],
                np.nan if row['chi2_dof'] is None else row['chi2_dof'],
                np.nan if row['n_points'] is None else row['n_points'],
                row['error_source'],
                str(analysis['path']),
            ]
            for row in rows
        ]
        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode='lines+markers',
                name=analysis['label'],
                line={'color': color, 'width': LINE_WIDTH},
                marker={'size': MARKER_SIZE, 'color': color},
                legendgroup=analysis['label'],
                error_y={
                    'type': 'data',
                    'array': np.where(visible_yerr, yerr, 0.0),
                    'visible': bool(np.any(visible_yerr)),
                    'thickness': 1,
                    'width': CAPSIZE,
                },
                customdata=customdata,
                hovertemplate=(
                    't<sub>lat</sub>=%{customdata[0]:g}<br>'
                    f'{CHI_HOVER_LABEL}=%{{y:.8g}}<br>'
                    'error=%{customdata[1]:.3g}<br>'
                    f'{CHI2_DOF_AXIS_TITLE}=%{{customdata[2]:.5g}}<br>'
                    'fit points=%{customdata[3]:.0f}<br>'
                    'error source=%{customdata[4]}<br>'
                    '%{customdata[5]}'
                    '<extra>' + analysis['label'] + '</extra>'
                ),
            ),
            row=1,
            col=1,
        )
        if np.any(visible_chi2):
            fig.add_trace(
                go.Scatter(
                    x=x[visible_chi2],
                    y=chi2_dof[visible_chi2],
                    mode='lines+markers',
                    name=analysis['label'] + ' ' + CHI2_DOF_AXIS_TITLE,
                    line={'color': color, 'width': LINE_WIDTH, 'dash': 'dot'},
                    marker={'size': MARKER_SIZE - 1, 'color': color, 'symbol': 'circle-open'},
                    legendgroup=analysis['label'],
                    showlegend=False,
                    customdata=[customdata[i] for i, is_visible in enumerate(visible_chi2) if is_visible],
                    hovertemplate=(
                        't<sub>lat</sub>=%{customdata[0]:g}<br>'
                        f'{CHI2_DOF_AXIS_TITLE}=%{{y:.5g}}<br>'
                        'fit points=%{customdata[3]:.0f}<br>'
                        'error source=%{customdata[4]}<br>'
                        '%{customdata[5]}'
                        '<extra>' + analysis['label'] + ' ' + CHI2_DOF_AXIS_TITLE + '</extra>'
                    ),
                ),
                row=2,
                col=1,
            )

    fig.update_layout(
        title=None,
        width=FIGURE_WIDTH,
        height=COMPARISON_FIGURE_HEIGHT,
        margin=PDF_MARGIN,
        showlegend=True,
        legend={'x': 0.98, 'xanchor': 'right', 'y': 0.98, 'yanchor': 'top'},
    )
    fig.update_xaxes(automargin=True, title_standoff=16, row=1, col=1)
    fig.update_xaxes(title_text=_axis_title(), automargin=True, title_standoff=16, row=2, col=1)
    fig.update_yaxes(title_text=CONTINUUM_CHI_AXIS_TITLE, automargin=True, row=1, col=1)
    fig.update_yaxes(title_text=CHI2_DOF_AXIS_TITLE, rangemode='tozero', automargin=True, row=2, col=1)
    if xlim is not None:
        fig.update_xaxes(range=list(xlim), row=1, col=1)
        fig.update_xaxes(range=list(xlim), row=2, col=1)
    if ylim is not None:
        fig.update_yaxes(range=list(ylim), row=1, col=1)
    return fig


fig = make_creutz_continuum_comparison_figure(analyses, xlim=XLIM, ylim=YLIM)
if SAVE_PDF:
    PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(PDF_PATH))
    print(f'Saved PDF: {PDF_PATH}')
fig